# MART_P01_CUSTO_VOLUME_MUNICIPIO

**P01. Qual o custo total e o volume de atendimentos (ambulatoriais e hospitalares) por
mês/ano em cada município de Alagoas?** *(dashboard)*

- Município = **local do atendimento** (onde o serviço foi efetuado).
- Ambulatorial: quantidade e valor **aprovados** do SIA, por mês de competência.
- Hospitalar: internações e valor total do SIH, por mês da internação.
- As duas fontes são agregadas separadamente e unidas por mês e município com
  `FULL OUTER JOIN`, para não perder municípios que só aparecem em uma delas.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P01_CUSTO_VOLUME_MUNICIPIO
    (ANO_MES, NR_ANO, NR_MES, COD_IBGE_6, NOME_MUNICIPIO, REGIAO_SAUDE, QTD_AMBULATORIAL,
     VL_AMBULATORIAL, QTD_INTERNACOES, VL_HOSPITALAR, QTD_TOTAL, VL_TOTAL)
WITH amb AS (
    SELECT t.ANO_MES, t.NR_ANO, t.NR_MES, f.SK_MUNICIPIO_ATENDIMENTO AS SK_MUN,
           SUM(f.QTD_APROVADA) AS QTD, SUM(f.VL_APROVADO) AS VL
    FROM FATO_PRODUCAO_AMBULATORIAL f
    JOIN DIM_TEMPO t ON t.SK_TEMPO = f.SK_TEMPO_COMPETENCIA
    GROUP BY t.ANO_MES, t.NR_ANO, t.NR_MES, f.SK_MUNICIPIO_ATENDIMENTO
),
hosp AS (
    SELECT t.ANO_MES, t.NR_ANO, t.NR_MES, f.SK_MUNICIPIO_ATENDIMENTO AS SK_MUN,
           SUM(f.QTD_INTERNACAO) AS QTD, SUM(f.VL_TOTAL) AS VL
    FROM FATO_INTERNACAO f
    JOIN DIM_TEMPO t ON t.SK_TEMPO = f.SK_TEMPO_INTERNACAO
    GROUP BY t.ANO_MES, t.NR_ANO, t.NR_MES, f.SK_MUNICIPIO_ATENDIMENTO
)
SELECT COALESCE(a.ANO_MES, h.ANO_MES), COALESCE(a.NR_ANO, h.NR_ANO), COALESCE(a.NR_MES, h.NR_MES),
       m.COD_IBGE_6, m.NOME_MUNICIPIO, m.REGIAO_SAUDE,
       NVL(a.QTD, 0), NVL(a.VL, 0), NVL(h.QTD, 0), NVL(h.VL, 0),
       NVL(a.QTD, 0) + NVL(h.QTD, 0), NVL(a.VL, 0) + NVL(h.VL, 0)
FROM amb a
FULL OUTER JOIN hosp h ON h.ANO_MES = a.ANO_MES AND h.SK_MUN = a.SK_MUN
JOIN DIM_MUNICIPIO m   ON m.SK_MUNICIPIO = COALESCE(a.SK_MUN, h.SK_MUN)
WHERE m.UF = 'AL'
  AND COALESCE(a.NR_ANO, h.NR_ANO) IN (SELECT DISTINCT TO_NUMBER(SUBSTR(ANO_MES_CMPT, 1, 4)) FROM FATO_INTERNACAO)
"""

In [ ]:
recarregar("MART_P01_CUSTO_VOLUME_MUNICIPIO", SQL, origem="FATO_PRODUCAO_AMBULATORIAL")

In [ ]:
consultar("SELECT * FROM MART_P01_CUSTO_VOLUME_MUNICIPIO FETCH FIRST 10 ROWS ONLY")